# 04 — Custom logits processor in vLLM (the EDT mechanism)

vLLM lets you register a **logits processor** that rescales/edits the logits at every decode
step, per request. This is the extension point the project's **Entropy-based Dynamic
Temperature (EDT)** sampling is built on: instead of a fixed temperature, scale the logits
by a temperature derived from the *entropy* of the current next-token distribution.

EDT formula (Zhang et al., 2024): `T = T0 * N**(theta / H)`, where `H` is the step entropy.
As `H → ∞` the exponent → 0 so `T → T0`; at confident (low-`H`) steps `N>1` raises `T`
(more diverse) and `N<1` lowers it (more conservative).

**Library focus:** `vllm.v1.sample.logits_processor` (`AdapterLogitsProcessor`).

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"

## The processor

`AdapterLogitsProcessor` gives a per-request hook. Returning `None` from
`new_req_logits_processor` leaves a request untouched, so one engine can mix EDT and
non-EDT requests — EDT is opted into via `SamplingParams(extra_args={"edt_mode": "token", ...})`.

In [ ]:
import torch
from vllm.v1.sample.logits_processor import (
    AdapterLogitsProcessor,
    RequestLogitsProcessor,
)

ENTROPY_FLOOR = 1e-6


def entropy(logits):  # Shannon entropy (nats) of softmax(logits)
    logp = torch.log_softmax(logits, dim=-1)
    return -(logp.exp() * logp).sum(dim=-1)


def edt_temperature(h, t0, n, theta):  # T0 * N ** (theta / H)
    return t0 * (n ** (theta / max(h, ENTROPY_FLOOR)))


class EDTLogitsProcessor(AdapterLogitsProcessor):
    def is_argmax_invariant(self) -> bool:
        return False  # rescaling changes the sampled distribution

    def new_req_logits_processor(self, params) -> RequestLogitsProcessor | None:
        ea = params.extra_args or {}
        if ea.get("edt_mode") != "token":
            return None  # inert unless the request opts in
        t0, n, theta = float(ea["T0"]), float(ea["N"]), float(ea["theta"])

        def _edt(output_ids, logits):
            h = float(entropy(logits))
            return logits / edt_temperature(h, t0, n, theta)

        return _edt

## Register it and compare

Pass the **class** at engine construction. EDT requests set `temperature=1.0` (the processor
does all the scaling) and carry the EDT params in `extra_args`. A plain request through the
same engine is unaffected.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL,
    dtype="auto",
    gpu_memory_utilization=0.9,
    logits_processors=[EDTLogitsProcessor],
)

prompt = "Write the opening line of a mystery novel:"

edt_args = {"edt_mode": "token", "T0": 0.9, "N": 0.8, "theta": 1.0}
edt_sp = SamplingParams(
    temperature=1.0, top_p=0.95, max_tokens=64, seed=0, extra_args=edt_args
)
fixed_sp = SamplingParams(temperature=0.9, top_p=0.95, max_tokens=64, seed=0)

print("EDT  :", llm.generate([prompt], edt_sp)[0].outputs[0].text.strip())
print("FIXED:", llm.generate([prompt], fixed_sp)[0].outputs[0].text.strip())

> **Project pointer:** this is exactly `llm_core.generation.temperature` (`entropy`,
> `edt_temperature`, `EDTLogitsProcessor`). The `token_edt` strategy in
> `llm_core.generation.generator.generate` wires it up; `seq_edt` instead estimates one
> temperature per sequence from a short warmup pass. Configs: `configs/gen/token_edt.yaml`.